# Quantile regression and conformal calibration

Fit quantile models with a chronological proper-training/calibration split. Report empirical coverage; temporal dependence means the usual exchangeable split-conformal guarantee does not automatically apply.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))


In [ ]:
X = df.copy()

X_train = X[X["tpep_pickup_datetime"].dt.month <= 8]
X_val = X[(X["tpep_pickup_datetime"].dt.month > 8)&(X["tpep_pickup_datetime"].dt.month <= 9)]
X_test = X[X["tpep_pickup_datetime"].dt.month > 9]
y_train = X_train.trip_count
y_val = X_val.trip_count
y_test = X_test.trip_count


In [ ]:
train = X_train.drop(columns = ["trip_count"])
val = X_val.drop(columns = ["trip_count"])
test = X_test.drop(columns = ["trip_count"])

train = train.drop(columns = ["tpep_pickup_datetime"])
val = val.drop(columns = ["tpep_pickup_datetime"])
test = test.drop(columns = ["tpep_pickup_datetime"])


In [ ]:
ablation = ["month"]
train_ablation = train.drop(columns = ablation)

categorical = ["PULocationID", "hours"]
numerical = [
    col for col in train_ablation.columns
    if (col not in categorical and col not in ablation)
]


In [ ]:
train_proper, train_calib, y_train_proper, y_train_calib = train_test_split(
    train_ablation,
    y_train,
    test_size = 0.25,
    shuffle = False
)


In [ ]:
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
model = make_pipeline(
    preprocessor,
    xgb.XGBRegressor(
        objective = "reg:quantileerror",
        quantile_alpha = np.array([0.05, 0.95]),
        tree_method = "hist",
        random_state = 42
    )
)

model.fit(train_proper, y_train_proper)


In [ ]:
quantiles = model.predict(train_calib)
lower = quantiles[:, 0]
upper = quantiles[:, 1]

score = [np.max((lower[j]-y_train_calib.iloc[j], y_train_calib.iloc[j]-upper[j])) for j in range(len(y_train_calib))]

# alpha = 0.10 matches the fitted 0.05 and 0.95 quantiles.
alpha = 0.10
n_calib = len(score)
quantile_level = (1 - alpha) * (1 + 1 / n_calib)

# Empirical inverse CDF: select an observed score, without interpolation.
score_quantile = np.quantile(score, quantile_level, method = "inverted_cdf")
print(f"Calibration size: {n_calib:,}")
print(f"Empirical quantile level: {quantile_level:.8f}")
print(f"Score quantile: {score_quantile:.6f}")


In [ ]:
quantiles_val = model.predict(val)

# Conformal adjustment of the validation quantile predictions.
lower_val = quantiles_val[:, 0] - score_quantile
upper_val = quantiles_val[:, 1] + score_quantile

prediction_intervals_val = pd.DataFrame(
    {"lower": lower_val, "upper": upper_val},
    index = val.index,
)
prediction_intervals_val


In [ ]:
def coverage(n_val, df, y):
    coverage = 0
    for i in range(n_val):
        if df.lower.iloc[i] <= y.iloc[i] and y.iloc[i] <= df.upper.iloc[i]:
            coverage += 1
    return coverage/n_val

coverage (len(val), prediction_intervals_val, y_val)


In [ ]:
def width(n_val, df):
    width = 0
    for i in range(n_val):
        width += df.upper.iloc[i]-df.lower.iloc[i]
    width = width/n_val
    return width

width(len(val), prediction_intervals_val)


In [ ]:
coverage


#### Comparison with uncalibrated quantile regression


In [ ]:
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
model_uncalib = make_pipeline(
    preprocessor,
    xgb.XGBRegressor(
        objective = "reg:quantileerror",
        quantile_alpha = np.array([0.05, 0.95]),
        tree_method = "hist",
        random_state = 42
    )
)

model_uncalib.fit(train_ablation, y_train)


In [ ]:
# Raw 5th/95th quantiles from the model fitted on all training rows.
quantiles_val_uncalib = model_uncalib.predict(val)
prediction_intervals_val_uncalib = pd.DataFrame(
    {"lower": quantiles_val_uncalib[:, 0], "upper": quantiles_val_uncalib[:, 1]},
    index = val.index,
)

# Also remove calibration from the proper-training model for a like-for-like comparison.
prediction_intervals_val_raw_proper = pd.DataFrame(
    {"lower": quantiles_val[:, 0], "upper": quantiles_val[:, 1]},
    index = val.index,
)

def interval_summary(name, intervals):
    assert intervals.index.equals(y_val.index)
    actual = y_val.to_numpy()
    lower_bound = intervals["lower"].to_numpy()
    upper_bound = intervals["upper"].to_numpy()
    return {
        "Method": name,
        "Coverage (%)": 100 * np.mean((lower_bound <= actual) & (actual <= upper_bound)),
        "Mean width": np.mean(upper_bound - lower_bound),
        "Crossed intervals": int(np.sum(lower_bound > upper_bound)),
    }

interval_comparison = pd.DataFrame([
    interval_summary("Proper training — uncalibrated", prediction_intervals_val_raw_proper),
    interval_summary("Proper training — calibrated CQR", prediction_intervals_val),
    interval_summary("All training — uncalibrated", prediction_intervals_val_uncalib),
]).set_index("Method")

coverage_uncalib = interval_comparison.loc["All training — uncalibrated", "Coverage (%)"] / 100
width_uncalib = interval_comparison.loc["All training — uncalibrated", "Mean width"]
print("September validation intervals — nominal coverage: 90%")
print(interval_comparison.to_string(float_format = lambda value: f"{value:.4f}"))
print("\nWidths are in pickups per zone per hour. Raw bounds are used without clipping or sorting.")
print("Crossed intervals have lower > upper; inspect these before interpreting mean width.")


#### Coverage for high, medium and low demand zones


In [ ]:
#Ranking the highest demand on an hourly average
rank = []
for z in X_train.PULocationID.unique():
    rank.append((z, X_train["trip_count"][X_train["PULocationID"] == z].mean()))

rank = sorted(rank, key = lambda x:x[1])
rank.reverse()

rank_zones = [rank[i][0] for i in range(len(rank))] #keep only the zone number
high_demand = rank_zones[:(263//3)]
mid_demand = rank_zones[(263//3):2*(263//3)]
low_demand = rank_zones[2*(263//3):]

val_high = val[val["PULocationID"].isin(high_demand)]
val_mid = val[val["PULocationID"].isin(mid_demand)]
val_low = val[val["PULocationID"].isin(low_demand)]

y_high = X_val.trip_count[X_val["PULocationID"].isin(high_demand)]
y_pred_high = model.predict(val_high)

y_mid = X_val.trip_count[X_val.PULocationID.isin(mid_demand)]
y__pred_mid = model.predict(val_mid)

y_low = X_val.trip_count[X_val.PULocationID.isin(low_demand)]
y_pred_low = model.predict(val_low)


In [ ]:
quantiles_high = model.predict(val_high)
quantiles_mid = model.predict(val_mid)
quantiles_low = model.predict(val_low)

# Conformal adjustment of the validation quantile predictions.
lower_val_high = quantiles_high[:, 0] - score_quantile
upper_val_high = quantiles_high[:, 1] + score_quantile

lower_val_mid = quantiles_mid[:, 0] - score_quantile
upper_val_mid = quantiles_mid[:, 1] + score_quantile

lower_val_low = quantiles_low[:, 0] - score_quantile
upper_val_low = quantiles_low[:, 1] + score_quantile


prediction_intervals_val_high = pd.DataFrame(
    {"lower": lower_val_high, "upper": upper_val_high},
    index = val_high.index,
)

prediction_intervals_val_mid = pd.DataFrame(
    {"lower": lower_val_mid, "upper": upper_val_mid},
    index = val_mid.index,
)

prediction_intervals_val_low = pd.DataFrame(
    {"lower": lower_val_low, "upper": upper_val_low},
    index = val_low.index,
)


In [ ]:
coverage(len(val_high), prediction_intervals_val_high, y_high), width(len(val_high), prediction_intervals_val_high) #High


In [ ]:
coverage(len(val_mid), prediction_intervals_val_mid, y_mid), width(len(val_mid), prediction_intervals_val_mid) #Mid


In [ ]:
coverage(len(val_low), prediction_intervals_val_low, y_low), width(len(val_low), prediction_intervals_val_low)#Low
